# EO-Benchmark Hub — EuroSAT Baseline (ResNet-18)

**Phase 1 official baseline notebook.** This notebook implements the full researcher-side workflow of EO-Benchmark Hub:

```
download EuroSAT → deterministic split (v1, seed=42) → train ResNet-18 on Colab GPU
→ inference on the hidden test split → export predictions.csv → validate locally → submit
```

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Colab-resilience built in:** checkpoints are saved to your Google Drive every epoch, and the training cell auto-resumes from the last checkpoint if your session disconnects. Just re-run all cells.

**Important — split consistency:** the split logic in this notebook (Section 3) is *identical* to the platform's `scripts/make_splits.py` (same seed, same stratification, same fractions). The platform evaluates you only on the `test_hidden` split, so predictions must cover exactly those `sample_id`s. Never train on `test_public` or `test_hidden`.

## 1. Configuration

In [ ]:
import os, sys, json, csv, random, shutil, zipfile, hashlib, time
from pathlib import Path

CFG = {
    # --- must match the platform's make_splits.py exactly ---
    "seed": 42,
    "split_version": "v1",
    "fractions": {"train": 0.70, "val": 0.10, "test_public": 0.10, "test_hidden": 0.10},

    # --- training hyperparameters (baseline) ---
    "img_size": 64,          # EuroSAT native resolution
    "batch_size": 128,
    "epochs": 10,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "label_smoothing": 0.05,

    # --- identity (goes into your model card) ---
    "benchmark": "eurosat",
    "model_name": "resnet18-baseline",
}

BENCHMARK_CLASSES = [  # canonical EuroSAT class names, alphabetical — the platform expects exactly these strings
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]

def set_seed(seed):
    random.seed(seed)
    try:
        import numpy as np; np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(CFG["seed"])
print("Config OK")

## 2. Environment setup

Mounts Google Drive (checkpoints + exported predictions survive disconnects) and installs the one extra dependency.

In [ ]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/eobench/eurosat")   # persistent (Drive)
    DATA_DIR = Path("/content/data")                              # ephemeral (fast local disk)
else:
    WORK_DIR = Path("./eobench_work/eurosat")
    DATA_DIR = Path("./eobench_data")

CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR  = WORK_DIR / "outputs"
for d in (CKPT_DIR, OUT_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

!pip -q install torchmetrics

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import torchmetrics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "  <-- WARNING: no GPU, training will be slow"))
print(f"Persistent work dir: {WORK_DIR}")

## 3. Download EuroSAT (RGB)

~90 MB, 27,000 images, 10 classes. Primary source is the official Zenodo record; the cell is idempotent (skips if already extracted).

In [ ]:
EUROSAT_URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"
ZIP_PATH = DATA_DIR / "EuroSAT_RGB.zip"
IMG_ROOT = DATA_DIR / "EuroSAT_RGB"   # -> EuroSAT_RGB/<Class>/<Class>_N.jpg

def download(url, dest, chunk=1 << 20):
    import urllib.request
    req = urllib.request.Request(url, headers={"User-Agent": "eobench-baseline/0.1"})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length", 0))
        done = 0
        while True:
            buf = r.read(chunk)
            if not buf:
                break
            f.write(buf); done += len(buf)
            if total:
                print(f"\r  {done/1e6:6.1f} / {total/1e6:.1f} MB", end="")
    print()

if not IMG_ROOT.exists():
    if not ZIP_PATH.exists():
        print("Downloading EuroSAT_RGB.zip ...")
        download(EUROSAT_URL, ZIP_PATH)
    print("Extracting ...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    ZIP_PATH.unlink(missing_ok=True)

# normalize the extraction root (some mirrors nest one folder deeper)
if not IMG_ROOT.exists():
    candidates = [p for p in DATA_DIR.rglob("AnnualCrop") if p.is_dir()]
    assert candidates, "Could not locate extracted EuroSAT class folders."
    IMG_ROOT = candidates[0].parent

n_imgs = sum(1 for _ in IMG_ROOT.rglob("*.jpg"))
found_classes = sorted(p.name for p in IMG_ROOT.iterdir() if p.is_dir())
assert found_classes == BENCHMARK_CLASSES, f"Class mismatch: {found_classes}"
print(f"OK — {n_imgs} images in {len(found_classes)} classes at {IMG_ROOT}")

## 4. Deterministic split — `split_v1`

Per-class stratified shuffle with a fixed RNG seed, sliced into `train / val / test_public / test_hidden` (70/10/10/10). The `sample_id` is the filename stem (e.g. `Forest_123`) — this is the ID the platform's ground truth uses.

The resulting `split_manifest_v1.csv` must be byte-identical to the one the platform publishes. If you already downloaded the official manifest from the benchmark page, the next cell verifies against it.

In [ ]:
def make_split_manifest(img_root: Path, classes, fractions: dict, seed: int):
    """Identical logic to the platform's scripts/make_splits.py — do not modify."""
    rows = []
    order = ["train", "val", "test_public", "test_hidden"]
    for cls in classes:                                # fixed class order
        files = sorted((img_root / cls).glob("*.jpg")) # deterministic file order
        rng = random.Random(f"{seed}-{cls}")            # per-class RNG -> class-independent
        idx = list(range(len(files)))
        rng.shuffle(idx)
        n = len(files)
        bounds, acc = [], 0.0
        for s in order[:-1]:
            acc += fractions[s]
            bounds.append(round(acc * n))
        splits = ([order[0]] * bounds[0]
                  + [order[1]] * (bounds[1] - bounds[0])
                  + [order[2]] * (bounds[2] - bounds[1])
                  + [order[3]] * (n - bounds[2]))
        for i, split in zip(idx, splits):
            f = files[i]
            rows.append({"sample_id": f.stem, "relpath": f"{cls}/{f.name}",
                         "label": cls, "split": split})
    rows.sort(key=lambda r: r["sample_id"])
    return rows

manifest = make_split_manifest(IMG_ROOT, BENCHMARK_CLASSES, CFG["fractions"], CFG["seed"])

MANIFEST_PATH = WORK_DIR / f"split_manifest_{CFG['split_version']}.csv"
with open(MANIFEST_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["sample_id", "relpath", "label", "split"])
    w.writeheader(); w.writerows(manifest)

from collections import Counter
counts = Counter(r["split"] for r in manifest)
print({k: counts[k] for k in ["train", "val", "test_public", "test_hidden"]})
print("Manifest written to", MANIFEST_PATH)

# fingerprint of the ID->split assignment; must match the value shown on the benchmark page
fp = hashlib.sha256("\n".join(f"{r['sample_id']},{r['split']}" for r in manifest).encode()).hexdigest()[:16]
print("Split fingerprint:", fp)

# Optional strict check against the official manifest downloaded from the platform:
OFFICIAL = WORK_DIR / "official_split_manifest_v1.csv"
if OFFICIAL.exists():
    official = {r["sample_id"]: r["split"] for r in csv.DictReader(open(OFFICIAL))}
    mine     = {r["sample_id"]: r["split"] for r in manifest}
    assert official == mine, "Split mismatch vs official manifest — do NOT submit."
    print("Verified against official manifest ✔")

## 5. Dataset & DataLoaders

In [ ]:
from PIL import Image

CLASS_TO_IDX = {c: i for i, c in enumerate(BENCHMARK_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# ImageNet normalization (we start from ImageNet-pretrained weights)
NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),        # satellite imagery has no canonical "up"
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    NORM,
])
eval_tf = transforms.Compose([transforms.ToTensor(), NORM])

class EuroSATSplit(Dataset):
    def __init__(self, manifest_rows, split, root, tf, with_labels=True):
        self.rows = [r for r in manifest_rows if r["split"] == split]
        self.root, self.tf, self.with_labels = root, tf, with_labels
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = self.tf(Image.open(self.root / r["relpath"]).convert("RGB"))
        if self.with_labels:
            return img, CLASS_TO_IDX[r["label"]]
        return img, r["sample_id"]

def loader(split, tf, shuffle, with_labels=True):
    ds = EuroSATSplit(manifest, split, IMG_ROOT, tf, with_labels)
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=True)

train_loader       = loader("train",       train_tf, shuffle=True)
val_loader         = loader("val",         eval_tf,  shuffle=False)
test_public_loader = loader("test_public", eval_tf,  shuffle=False)
print(f"train={len(train_loader.dataset)}  val={len(val_loader.dataset)}  "
      f"test_public={len(test_public_loader.dataset)}")

## 6. Model — ResNet-18 (ImageNet-pretrained)

In [ ]:
def build_model():
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, len(BENCHMARK_CLASSES))
    return m.to(DEVICE)

model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 ready — {n_params/1e6:.2f}M params  (record this in your model card)")

## 7. Training — with checkpoint/auto-resume

Every epoch saves `last.pt` to Drive; the best validation accuracy saves `best.pt`. If Colab disconnects, just **Runtime → Run all** — this cell picks up where it left off.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

acc_metric = torchmetrics.Accuracy(task="multiclass", num_classes=len(BENCHMARK_CLASSES)).to(DEVICE)

LAST_CKPT, BEST_CKPT = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"
start_epoch, best_val_acc = 0, 0.0

if LAST_CKPT.exists():                                   # ---- auto-resume ----
    ck = torch.load(LAST_CKPT, map_location=DEVICE)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optim"])
    scheduler.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, best_val_acc = ck["epoch"] + 1, ck["best_val_acc"]
    print(f"Resumed from epoch {ck['epoch']} (best val acc so far: {best_val_acc:.4f})")

@torch.no_grad()
def evaluate(loader):
    model.eval(); acc_metric.reset()
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        acc_metric.update(model(x).argmax(1), y)
    return acc_metric.compute().item()

for epoch in range(start_epoch, CFG["epochs"]):
    model.train(); t0 = time.time(); running = 0.0
    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item()
        if step % 50 == 0:
            print(f"\r  epoch {epoch+1}/{CFG['epochs']}  step {step}/{len(train_loader)}"
                  f"  loss {running/(step+1):.4f}", end="")
    scheduler.step()
    val_acc = evaluate(val_loader)
    print(f"\repoch {epoch+1}/{CFG['epochs']}  train_loss {running/len(train_loader):.4f}"
          f"  val_acc {val_acc:.4f}  ({time.time()-t0:.0f}s)          ")

    state = {"epoch": epoch, "model": model.state_dict(), "optim": optimizer.state_dict(),
             "sched": scheduler.state_dict(), "scaler": scaler.state_dict(),
             "best_val_acc": max(best_val_acc, val_acc), "cfg": CFG}
    torch.save(state, LAST_CKPT)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(state, BEST_CKPT)
        print(f"  ↑ new best ({best_val_acc:.4f}) — saved best.pt")

print(f"\nTraining done. Best val accuracy: {best_val_acc:.4f}")

## 8. Sanity check on `test_public`

Labels for `test_public` are released precisely so you can estimate your hidden-split score *before* spending a submission. Expect the hidden score to land within ~±1% of this.

In [ ]:
ck = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ck["model"])
pub_acc = evaluate(test_public_loader)
print(f"test_public accuracy (best checkpoint): {pub_acc:.4f}")

# per-class F1 on test_public — useful for the model card notes
f1 = torchmetrics.F1Score(task="multiclass", num_classes=len(BENCHMARK_CLASSES), average=None).to(DEVICE)
model.eval()
with torch.no_grad():
    for x, y in test_public_loader:
        f1.update(model(x.to(DEVICE)).argmax(1), y.to(DEVICE))
for cls, s in zip(BENCHMARK_CLASSES, f1.compute().tolist()):
    print(f"  {cls:<22s} F1 = {s:.4f}")

## 9. Inference on `test_hidden` → `predictions.csv`

Exact schema required by the platform's scene-classification evaluator:

| column | type | constraint |
|---|---|---|
| `sample_id` | string | exactly the `test_hidden` IDs from the manifest, each exactly once |
| `predicted_label` | string | one of the 10 canonical class names |

In [ ]:
hidden_loader = loader("test_hidden", eval_tf, shuffle=False, with_labels=False)

predictions = []
model.eval()
with torch.no_grad():
    for x, sample_ids in hidden_loader:
        preds = model(x.to(DEVICE, non_blocking=True)).argmax(1).cpu().tolist()
        predictions += [(sid, IDX_TO_CLASS[p]) for sid, p in zip(sample_ids, preds)]

predictions.sort(key=lambda t: t[0])
PRED_PATH = OUT_DIR / f"predictions_{CFG['model_name']}_{CFG['split_version']}.csv"
with open(PRED_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["sample_id", "predicted_label"])
    w.writerows(predictions)

print(f"Wrote {len(predictions)} predictions -> {PRED_PATH}")

## 10. Local validation (mirror of the platform's checks)

Runs the same checks the platform's sync validator runs, so a rejected upload never costs you a quota slot.

In [ ]:
def validate_predictions(pred_path, manifest_rows, classes):
    errors = []
    expected = {r["sample_id"] for r in manifest_rows if r["split"] == "test_hidden"}
    with open(pred_path, newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != ["sample_id", "predicted_label"]:
            return [f"Bad header: {reader.fieldnames} (expected ['sample_id', 'predicted_label'])"]
        rows = list(reader)

    ids = [r["sample_id"] for r in rows]
    seen = set(ids)
    if len(ids) != len(seen):
        errors.append(f"{len(ids) - len(seen)} duplicate sample_id(s)")
    missing, extra = expected - seen, seen - expected
    if missing: errors.append(f"{len(missing)} missing test_hidden id(s), e.g. {sorted(missing)[:3]}")
    if extra:   errors.append(f"{len(extra)} unknown id(s) (not in test_hidden), e.g. {sorted(extra)[:3]}")

    valid = set(classes)
    bad = {r["predicted_label"] for r in rows} - valid
    if bad: errors.append(f"Invalid label value(s): {sorted(bad)}")
    return errors

errs = validate_predictions(PRED_PATH, manifest, BENCHMARK_CLASSES)
if errs:
    print("VALIDATION FAILED — fix before submitting:")
    for e in errs: print("  ✗", e)
else:
    print("✔ predictions.csv is valid — ready to submit")
    print("  sha256:", hashlib.sha256(open(PRED_PATH, 'rb').read()).hexdigest()[:16])

## 11. Submit

**Option A — web UI (default):** download the CSV, then on your EO-Benchmark Hub instance go to *EuroSAT → Submit*, upload it, and fill in the model card.

**Option B — API (only if your local instance is exposed via ngrok/cloudflared):** fill in the two variables and run the second cell.

In [ ]:
# Option A: download the file to your machine
if IN_COLAB:
    from google.colab import files
    files.download(str(PRED_PATH))
else:
    print("Predictions file at:", PRED_PATH.resolve())

In [ ]:
# Option B: direct API submission (requires a public tunnel to your local instance)
API_BASE  = ""   # e.g. "https://abc123.ngrok-free.app"  — leave empty to skip
API_TOKEN = ""   # personal token from your profile page

if API_BASE and API_TOKEN:
    import requests
    H = {"Authorization": f"Bearer {API_TOKEN}"}
    with open(PRED_PATH, "rb") as f:
        r = requests.post(f"{API_BASE}/benchmarks/{CFG['benchmark']}/submissions",
                          headers=H, files={"file": (PRED_PATH.name, f, "text/csv")},
                          data={"file_format": "csv"})
    r.raise_for_status()
    sub = r.json()
    print("Submission id:", sub["id"], "| status:", sub["status"])
    print(f"Track it at {API_BASE}/submissions/{sub['id']}  (or GET the same path on the API)")
else:
    print("API_BASE/API_TOKEN not set — use Option A (web upload).")

## 12. Model card — copy-paste starter

Fill the platform's model card form with (values printed by earlier cells):

- **Architecture:** ResNet-18 (torchvision), final FC replaced for 10 classes
- **Params:** ~11.2 M · **Input size:** 64×64 RGB
- **Pretraining:** ImageNet-1k (`IMAGENET1K_V1`) · **External data:** no
- **Training:** AdamW lr 3e-4, wd 1e-4, cosine schedule, 10 epochs, batch 128, label smoothing 0.05, AMP; augmentations: H/V flip, ±15° rotation, light color jitter
- **Hardware:** Google Colab (T4) · **Training time:** ~*(fill from epoch logs)*
- **Split:** `split_v1`, seed 42 · **Notebook:** link this notebook (File → Save a copy in GitHub)

### Ideas to beat the baseline
Try in roughly this order: more epochs + ResNet-50 → stronger augmentation (RandAugment, MixUp/CutMix) → test-time augmentation (average logits over flips) → EO-specific pretrained backbones (SSL4EO, SatMAE) instead of ImageNet.